# GONG + CHASE shape synthesis — fresh best84 recipe on T4

Run the single cell below in a Colab **T4 GPU** runtime. The first run copies a complete, checksum-verified code snapshot to Drive. Run the **same cell** after interruption or a runtime reset: it resumes from the last completed epoch using the saved code without requesting GitHub again. Interrupted epochs are replayed.

- Fresh ImageNet ConvNeXt-Tiny encoder + new U-Net decoder. Original best84 weights are used only for comparison on the existing fixed validation set.
- **100-epoch schedule**: original best84 trained for 100 epochs and selected epoch 84. T4 uses FP16, batch 1 × accumulation 4, 400 crops/epoch and workers 0. No YOLO training.
- Real GONG training images plus CHASE-derived shapes rendered on your external `clean_solar.jpg`. The synthetic sampling probability is **50% during epochs 4–94** (original best84: 20%); epochs 1–3 and 95–100 use only real GONG crops. The clean background is recorded as user-supplied and reported filament-free; its source URL was not provided. Original data and weights remain intact. The source clean image is read without overwriting it. Model checkpoints and small review outputs stay on Drive; generated image caches stay on local Colab disk.
- `PHASE="prepare"` produces synthesis previews on CPU; `PHASE="all"` prepares then trains. A smaller `RUN_UNTIL_EPOCH` pauses at that completed epoch while retaining the 100-epoch learning-rate schedule.
- Full logs: `OUTPUT/training.log`. The notebook prints a small tail every 30 seconds. The result ZIP excludes model weights and full image/RLE caches.

This launcher has CPU fixture tests; full T4 training must run in Colab. Validation tuning does not guarantee a leaderboard improvement.


In [ ]:
# One cell: first launch, pause/resume, or a fresh Colab runtime.
PHASE = "all"                 # "prepare" makes previews without GPU training.
RUN_UNTIL_EPOCH = 100          # A stage cap; the learning-rate schedule remains 100.
from pathlib import Path
from google.colab import drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
CLEAN_BACKGROUND = BASE / "clean_solar.jpg"  # External image supplied by you.
CLEAN_BACKGROUND_PROVENANCE = "external_user_supplied"
OUTPUT = BASE / "fixed_split_experiments/chase_shape_best84_recipe_t4_v1"
CACHE_ROOT = Path("/content/chase_shape_best84_recipe_t4_cache")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
CHECKOUT = Path("/content/solar_chase_shape_github")
drive.mount("/content/drive")
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
state_path = OUTPUT / "bootstrap_state.json"
if state_path.is_file():
    # No GitHub request on resume. Verify before executing the saved bootstrap.
    state = json.loads(state_path.read_text())
    if state.get("scope") != "chase_shape_best84_recipe_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("This saved bootstrap belongs to another experiment.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed; existing files were preserved.")
    manifest = json.loads(manifest_path.read_text())
    manifest_identity = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(manifest_identity).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    bootstrap = snapshot / "tools/chase_shape_colab.py"
    if bootstrap.is_symlink() or not bootstrap.resolve().is_relative_to(snapshot.resolve()):
        raise ValueError("Unsafe saved bootstrap path.")
    if sha(bootstrap) != manifest.get("tools/chase_shape_colab.py"):
        raise ValueError("Saved launcher changed; existing files were preserved.")
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True).strip()
        if origin != REPOSITORY or dirty:
            raise ValueError("Existing checkout differs or has local edits. Preserve it and choose a different CHECKOUT.")
        subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
    bootstrap = CHECKOUT / "tools/chase_shape_colab.py"
    if not bootstrap.is_file():
        raise FileNotFoundError("The selected repository does not contain the CHASE-shape launcher.")
command = [sys.executable, "-u", str(bootstrap), "--base", str(BASE),
           "--output", str(OUTPUT), "--cache-root", str(CACHE_ROOT),
           "--clean-background", str(CLEAN_BACKGROUND),
           "--clean-background-provenance", CLEAN_BACKGROUND_PROVENANCE,
           "--phase", PHASE, "--run-until", str(RUN_UNTIL_EPOCH)]
process = subprocess.Popen(command, start_new_session=True)
try:
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
result = json.loads((OUTPUT / "latest_result.json").read_text())
print("Status:", result.get("status"), "Completed epoch:", result.get("epochs_completed", 0))
if result.get("archive") and result.get("status") in ("complete", "completed", "paused"):
    archive = Path(result["archive"]).resolve()
    if not archive.is_relative_to(OUTPUT) or not archive.is_file() or sha(archive) != result.get("archive_sha256"):
        raise ValueError("Result ZIP is missing or changed; it was not downloaded.")
    print("Review ZIP:", archive)
    files.download(str(archive))
else:
    print("Preparation/preview results:", OUTPUT / "synthesis")
